# Mind2vec Results Visualization

This notebook visualizes pre-computed CHISCo results for the paper. Update the values in the **Results** section and run the notebook to regenerate the figures.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_ORDER = ["Closed-set", "Contrastive"]

mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8.5, "axes.titlesize": 9.5,
                     "axes.labelsize": 8.5, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
                     "legend.fontsize": 7.5, "axes.linewidth": 0.8, "lines.linewidth": 1.35,
                     "pdf.fonttype": 42, "ps.fonttype": 42})

def save_pdf(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(path, format="pdf", bbox_inches="tight", facecolor="white")
    print(f"Saved: {path}")

## Results

Enter the results from the evaluation code here. The plotting cells below use these values directly.


In [ ]:
# Enter or replace the pre-computed results below.
SUBJECTS = ["S1", "S2", "S3", "S4", "S5"]
TOPK_METRICS = ["Top-1", "Top-2", "Top-3", "Top-5", "Top-10"]

RESULTS = {
    "Closed-set": {
        "topk_mean": [
            [0.100, 0.163, 0.217, 0.305, 0.477],
            [0.108, 0.174, 0.230, 0.324, 0.502],
            [0.100, 0.176, 0.232, 0.321, 0.492],
            [0.096, 0.158, 0.217, 0.301, 0.484],
            [0.095, 0.163, 0.220, 0.312, 0.485],
        ],
        "topk_sd": [
            [0.013, 0.015, 0.018, 0.021, 0.037],
            [0.011, 0.015, 0.010, 0.017, 0.015],
            [0.010, 0.011, 0.009, 0.015, 0.009],
            [0.005, 0.008, 0.016, 0.016, 0.021],
            [0.005, 0.003, 0.009, 0.018, 0.016],
        ],
        "macro_f1": [
            (0.061, 0.013),
            (0.065, 0.013),
            (0.063, 0.012),
            (0.066, 0.005),
            (0.059, 0.005),
        ],
        "two_vs_two": [
            (0.695, 0.026),
            (0.717, 0.017),
            (0.716, 0.007),
            (0.708, 0.010),
            (0.698, 0.011),
        ],
    },

    "Contrastive": {
        "topk_mean": [
            [0.083, 0.148, 0.199, 0.292, 0.467],
            [0.100, 0.170, 0.229, 0.327, 0.499],
            [0.090, 0.155, 0.211, 0.302, 0.481],
            [0.096, 0.167, 0.217, 0.311, 0.481],
            [0.091, 0.161, 0.216, 0.309, 0.488],
        ],
        "topk_sd": [
            [0.007, 0.003, 0.010, 0.008, 0.010],
            [0.004, 0.005, 0.010, 0.011, 0.015],
            [0.009, 0.014, 0.017, 0.021, 0.021],
            [0.005, 0.006, 0.012, 0.004, 0.008],
            [0.010, 0.009, 0.013, 0.015, 0.018],
        ],
        "macro_f1": [
            (0.061, 0.007),
            (0.070, 0.006),
            (0.057, 0.010),
            (0.059, 0.010),
            (0.066, 0.009),
        ],
        "two_vs_two": [
            (0.702, 0.006),
            (0.726, 0.009),
            (0.703, 0.017),
            (0.713, 0.005),
            (0.721, 0.010),
        ],
    },
}

# Prior-aware random Top-k baseline, one row per subject.
TOPK_RANDOM = [
    [0.029, 0.059, 0.088, 0.146, 0.290],
    [0.030, 0.059, 0.088, 0.147, 0.290],
    [0.030, 0.060, 0.090, 0.149, 0.295],
    [0.029, 0.059, 0.088, 0.146, 0.290],
    [0.030, 0.060, 0.089, 0.148, 0.292],
]

MACRO_F1_RANDOM = 1 / 39
TWO_VS_TWO_CHANCE = 0.50

# Semantic-axis results (entered as percentages).
SEMANTIC_AXIS_DATA = np.array([
    [61.6, 58.6, 58.6, 52.9, 52.8],
    [61.9, 58.2, 60.9, 53.3, 52.9],
    [61.3, 59.2, 59.0, 52.5, 52.2],
    [61.2, 60.1, 60.3, 54.0, 52.6],
    [60.4, 60.1, 61.2, 53.9, 54.2],
]) / 100

SEMANTIC_AXIS_LABELS = [
    "Practical ↔ Emotional",
    "Physical ↔ Cognitive",
    "Goal-directed ↔ Leisure",
    "Mobility ↔ Personal life",
    "Individual ↔ Social",
]

## Top-1 and Top-5 accuracy

Subject-wise comparison of the closed-set and contrastive models. Error bars show SD, and the dotted references indicate the random baselines.


In [ ]:
idx_top1, idx_top5 = (0, 3)
x = np.arange(len(SUBJECTS)) * 0.22
off_closed, off_contr = (-0.05, 0.05)
FS_LABEL, FS_TICK, FS_LEGEND, FS_ANN = (13, 11, 10.5, 9.5)
TOP1_COLOR = '#c65b7c'
TOP5_COLOR = '#2a9d8f'
CONNECTOR_COLOR = '#cfcfcf'
BASELINE_ALPHA = 0.1
MARKER_CLOSED = 'o'
MARKER_CONTR = 'D'
MS_CLOSED, MS_CONTR = (7.5, 7.2)
fig, ax = plt.subplots(figsize=(6.6, 3.6))
closed_top1 = np.asarray(RESULTS['Closed-set']['topk_mean'])[:, idx_top1]
closed_top1_sd = np.asarray(RESULTS['Closed-set']['topk_sd'])[:, idx_top1]
contr_top1 = np.asarray(RESULTS['Contrastive']['topk_mean'])[:, idx_top1]
contr_top1_sd = np.asarray(RESULTS['Contrastive']['topk_sd'])[:, idx_top1]
closed_top5 = np.asarray(RESULTS['Closed-set']['topk_mean'])[:, idx_top5]
closed_top5_sd = np.asarray(RESULTS['Closed-set']['topk_sd'])[:, idx_top5]
contr_top5 = np.asarray(RESULTS['Contrastive']['topk_mean'])[:, idx_top5]
contr_top5_sd = np.asarray(RESULTS['Contrastive']['topk_sd'])[:, idx_top5]
# Random baselines
baseline_top1 = np.asarray(TOPK_RANDOM)[:, idx_top1]
baseline_top5 = np.asarray(TOPK_RANDOM)[:, idx_top5]
b1_mean, b1_min, b1_max = (baseline_top1.mean(), baseline_top1.min(), baseline_top1.max())
b5_mean, b5_min, b5_max = (baseline_top5.mean(), baseline_top5.min(), baseline_top5.max())
ax.axhspan(b1_min, b1_max, color=TOP1_COLOR, alpha=BASELINE_ALPHA, zorder=0)
ax.axhspan(b5_min, b5_max, color=TOP5_COLOR, alpha=BASELINE_ALPHA, zorder=0)
ax.axhline(b1_mean, color=TOP1_COLOR, linestyle=(0, (2, 2)), linewidth=1.3, zorder=1)
ax.axhline(b5_mean, color=TOP5_COLOR, linestyle=(0, (2, 2)), linewidth=1.3, zorder=1)


# Plot subject-wise results
def add_value_label(ax, x0, y0, err, color, pad=0.012):
    """Place a vertical percentage label just above an error bar."""
    ax.text(x0, y0 + err + pad, f'{100 * y0:.1f}%', rotation=90, ha='center', va='bottom', fontsize=FS_ANN, color=color, fontweight='medium', alpha=0.95, clip_on=False, zorder=6)
for i, x_center in enumerate(x):
    x_closed = x_center + off_closed
    x_contr = x_center + off_contr
    ax.plot([x_closed, x_contr], [closed_top1[i], contr_top1[i]], color=CONNECTOR_COLOR, linewidth=1.6, zorder=2)
    ax.plot([x_closed, x_contr], [closed_top5[i], contr_top5[i]], color=CONNECTOR_COLOR, linewidth=1.6, zorder=2)
    ax.errorbar(x_closed, closed_top1[i], yerr=closed_top1_sd[i], fmt=MARKER_CLOSED, color=TOP1_COLOR, markersize=MS_CLOSED, markeredgecolor='white', markeredgewidth=0.9, elinewidth=1.2, capsize=3, capthick=1.1, linewidth=0, zorder=3)
    ax.errorbar(x_contr, contr_top1[i], yerr=contr_top1_sd[i], fmt=MARKER_CONTR, color=TOP1_COLOR, markersize=MS_CONTR, markeredgecolor='white', markeredgewidth=0.9, elinewidth=1.2, capsize=3, capthick=1.1, linewidth=0, zorder=4)
    ax.errorbar(x_closed, closed_top5[i], yerr=closed_top5_sd[i], fmt=MARKER_CLOSED, color=TOP5_COLOR, markersize=MS_CLOSED, markeredgecolor='white', markeredgewidth=0.9, elinewidth=1.2, capsize=3, capthick=1.1, linewidth=0, zorder=3)
    ax.errorbar(x_contr, contr_top5[i], yerr=contr_top5_sd[i], fmt=MARKER_CONTR, color=TOP5_COLOR, markersize=MS_CONTR, markeredgecolor='white', markeredgewidth=0.9, elinewidth=1.2, capsize=3, capthick=1.1, linewidth=0, zorder=4)
    add_value_label(ax, x_closed, closed_top1[i], closed_top1_sd[i], TOP1_COLOR)
    add_value_label(ax, x_contr, contr_top1[i], contr_top1_sd[i], TOP1_COLOR)
    add_value_label(ax, x_closed, closed_top5[i], closed_top5_sd[i], TOP5_COLOR)
    add_value_label(ax, x_contr, contr_top5[i], contr_top5_sd[i], TOP5_COLOR)

# Styling and export
ax.set_ylabel('Accuracy', fontsize=FS_LABEL, labelpad=8)
ax.set_xlabel('Subject', fontsize=FS_LABEL, labelpad=6)
ax.set_xticks(x)
ax.set_xticklabels(SUBJECTS, fontsize=FS_TICK)
ax.set_xlim(x[0] - 0.09, x[-1] + 0.09)
ax.set_ylim(0, 0.46)
ax.set_yticks(np.arange(0, 0.46, 0.1))
ax.tick_params(axis='x', labelsize=FS_TICK, width=1.1, length=4)
ax.tick_params(axis='y', labelsize=FS_TICK, width=1.1, length=4)
ax.grid(axis='y', color='#e8e8e8', linewidth=0.9)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.1)
ax.spines['bottom'].set_linewidth(1.1)
legend_handles = [mpl.lines.Line2D([0], [0], marker=MARKER_CLOSED, color='none', markerfacecolor='gray', markeredgecolor='white', markeredgewidth=0.9, markersize=8, label='Closed-set'), mpl.lines.Line2D([0], [0], marker=MARKER_CONTR, color='none', markerfacecolor='gray', markeredgecolor='white', markeredgewidth=0.9, markersize=7.7, label='Contrastive'), mpl.lines.Line2D([0], [0], color=TOP1_COLOR, marker='o', markersize=7, linewidth=1.6, label='Top-1'), mpl.lines.Line2D([0], [0], color=TOP5_COLOR, marker='o', markersize=7, linewidth=1.6, label='Top-5'), mpl.lines.Line2D([0], [0], color=TOP1_COLOR, linestyle=(0, (2, 2)), linewidth=1.3, label='Top-1 random'), mpl.lines.Line2D([0], [0], color=TOP5_COLOR, linestyle=(0, (2, 2)), linewidth=1.3, label='Top-5 random')]
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.95), ncol=3, frameon=False, columnspacing=1.25, handlelength=1.8, handletextpad=0.6, fontsize=FS_LEGEND)
fig.subplots_adjust(left=0.1, right=0.995, bottom=0.18, top=0.81)
save_pdf(fig, 'top1_top5_combined.pdf')
plt.show()

## Macro-F1 and 2v2 accuracy

Subject-wise Macro-F1 and 2v2 results for both models. Error bars show SD and dotted lines indicate the corresponding random/chance baselines.


In [ ]:
FS_LABEL, FS_TICK, FS_LEGEND, FS_ANN, FS_TITLE = (13, 11, 10.5, 9.5, 13)
POINT_COLOR = '#4a4a4a'
BASELINE_COLOR = '#7a7a7a'
CONNECTOR_COLOR = '#cfcfcf'
MARKERS = {'Closed-set': 'o', 'Contrastive': 'D'}
MS = {'Closed-set': 7.5, 'Contrastive': 7.2}
x = np.arange(len(SUBJECTS)) * 0.22
offsets = {'Closed-set': -0.05, 'Contrastive': 0.05}


# Plot one metric panel
def plot_metric(ax, metric, title, ylim, yticks, baseline, baseline_label, ann_pad):
    """Plot one paired subject-level metric panel."""
    data = {model: np.asarray(RESULTS[model][metric]) for model in MODEL_ORDER}
    ax.axhline(baseline, color=BASELINE_COLOR, linestyle=(0, (2, 2)), linewidth=1.3, zorder=1)
    ax.text(0.985, baseline + 0.018 * (ylim[1] - ylim[0]), baseline_label, transform=ax.get_yaxis_transform(), ha='right', va='bottom', fontsize=FS_ANN, color=BASELINE_COLOR, fontweight='medium')
    for i, x_center in enumerate(x):
        closed = data['Closed-set']
        contrastive = data['Contrastive']
        ax.plot([x_center + offsets['Closed-set'], x_center + offsets['Contrastive']], [closed[i, 0], contrastive[i, 0]], color=CONNECTOR_COLOR, linewidth=1.6, zorder=2)
        for model in MODEL_ORDER:
            mean, sd = data[model][i]
            x_point = x_center + offsets[model]
            ax.errorbar(x_point, mean, yerr=sd, fmt=MARKERS[model], color=POINT_COLOR, markersize=MS[model], markeredgecolor='white', markeredgewidth=0.9, elinewidth=1.2, capsize=3, capthick=1.1, linewidth=0, zorder=3)
            ax.text(x_point, mean + sd + ann_pad, f'{100 * mean:.1f}%', rotation=90, ha='center', va='bottom', fontsize=FS_ANN, color=POINT_COLOR, fontweight='medium', clip_on=False, zorder=6)
    ax.set_title(title, fontsize=FS_TITLE, weight='bold', pad=16)
    ax.set_xticks(x, SUBJECTS)
    ax.set_xlim(x[0] - 0.15, x[-1] + 0.15)
    ax.set_ylim(*ylim)
    ax.set_yticks(yticks)
    ax.tick_params(axis='both', labelsize=FS_TICK, width=1.1, length=4)
    ax.grid(axis='y', color='#e8e8e8', linewidth=0.9)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(1.1)

# Create the two-panel figure
fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.6))
plot_metric(axes[0], 'macro_f1', 'Macro-F1', (0.015, 0.095), np.arange(0.02, 0.1, 0.01), MACRO_F1_RANDOM, 'Random = 2.5%', 0.0025)
plot_metric(axes[1], 'two_vs_two', '2v2 accuracy', (0.48, 0.78), np.arange(0.5, 0.76, 0.05), TWO_VS_TWO_CHANCE, 'Chance = 50%', 0.01)
axes[0].set_ylabel('Macro-F1', fontsize=FS_LABEL, labelpad=8)
axes[1].set_ylabel('2v2 accuracy', fontsize=FS_LABEL, labelpad=8)
legend_handles = [Line2D([0], [0], marker=MARKERS[model], color='none', markerfacecolor='black', markeredgecolor='white', markeredgewidth=0.9, markersize=8, label=model) for model in MODEL_ORDER]
fig.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, 0.99), ncol=2, frameon=False, columnspacing=1.5, handletextpad=0.6, fontsize=FS_LEGEND)
fig.supxlabel('Subject', fontsize=FS_LABEL, y=0.035)
fig.subplots_adjust(left=0.09, right=0.995, bottom=0.2, top=0.79, wspace=0.28)
save_pdf(fig, 'macro_f1_2v2.pdf')
plt.show()

## Generalization across semantic axes

Balanced accuracy across the five semantic contrasts. Bars show the mean across subjects and error bars show the across-subject SD.


In [ ]:
semantic_mean = SEMANTIC_AXIS_DATA.mean(axis=0)
semantic_sd = SEMANTIC_AXIS_DATA.std(axis=0, ddof=1)
x = np.arange(len(SEMANTIC_AXIS_LABELS))

# Plot mean ± SD across subjects
plt.rcParams.update({'font.size': 14, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 1, 'pdf.fonttype': 42, 'ps.fonttype': 42})
fig, ax = plt.subplots(figsize=(11, 6.5))
ax.bar(x, semantic_mean, width=0.56, color='#5B8DB8', edgecolor='#3F6F96', linewidth=0, zorder=3, alpha=0.8)
ax.errorbar(x, semantic_mean, yerr=semantic_sd, fmt='none', ecolor='#333333', elinewidth=1.5, capsize=5, capthick=1.5, zorder=4)
ax.axhline(0.5, linestyle='--', color='#666666', linewidth=1.5, label='Chance (50%)', zorder=2)
for i, (mean_value, sd_value) in enumerate(zip(semantic_mean, semantic_sd)):
    ax.text(i, mean_value + sd_value + 0.008, f'{mean_value:.1%}', ha='center', va='bottom', fontsize=13, fontweight='medium')
ax.set_title('Generalization across semantic axes', fontsize=19, pad=15)
ax.set_ylabel('Balanced accuracy', fontsize=16)
ax.set_xticks(x)
ax.set_xticklabels(SEMANTIC_AXIS_LABELS, rotation=22, ha='right', fontsize=13)
ax.tick_params(axis='y', labelsize=13, length=0)
ax.tick_params(axis='x', length=0)
ax.set_ylim(0, 0.72)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(axis='y', linestyle='-', linewidth=0.7, alpha=0.15, zorder=0)
ax.legend(frameon=False, fontsize=13, loc='upper right')
ax.spines['left'].set_color('#888888')
ax.spines['bottom'].set_color('#888888')
plt.tight_layout()

# Export
semantic_pdf = OUTPUT_DIR / 'semantic_axes_generalization.pdf'
semantic_png = OUTPUT_DIR / 'semantic_axes_generalization.png'
plt.savefig(semantic_pdf, bbox_inches='tight')
plt.savefig(semantic_png, dpi=600, bbox_inches='tight')
print(f'Saved: {semantic_pdf}')
print(f'Saved: {semantic_png}')
plt.show()